In [79]:
import pandas as pd
import requests


In [80]:
url = "https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=1"
response = requests.get(url)
data = response.json()


In [81]:
print(data.keys())
print(len(data['results']))


dict_keys(['page', 'results', 'total_pages', 'total_results'])
20


In [82]:
movie = data['results'][0]
print(movie.keys())

dict_keys(['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count'])


In [83]:
titles = []
descriptions = []
genres = []
for page in range(1,472):
  url = f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={page}"
  response = requests.get(url)
  data = response.json()
  for movie in data['results']:
    titles.append(movie['title'])
    descriptions.append(movie['overview'])
    genres.append(movie['genre_ids'])
print(len(titles), len(descriptions), len(genres))
print(titles[0])
print(titles[20])


9420 9420 9420
Batman: Knightfall Part 1: Knightfall
Dilwale Dulhania Le Jayenge


In [84]:
df =pd.DataFrame({
    'movie_name':titles,
    'description':descriptions,
    'genre_id':genres
})
print(df.shape)
df.sample(10)


(9420, 3)


,movie_name,description,genre_id
208,Wonder,The story of August Pullman – a boy with facia...,"[10751, 18]"
979,Amores Perros,A fatalistic car crash in Mexico city sets off...,"[18, 53]"
4755,The Best Is Yet to Come,"Following a huge misunderstanding, two friends...","[35, 18]"
2769,Only Lovers Left Alive,A depressed musician reunites with his lover. ...,"[18, 10749, 14]"
3138,Baby Boy,"The story of Jody, a misguided, 20-year-old Af...","[80, 18, 10749, 53]"
2144,Now You See Me,An FBI agent and an Interpol detective track a...,"[53, 80]"
8579,V/H/S/94,"After the discovery of a mysterious VHS tape, ...","[27, 53]"
7215,The Program,An Irish sports journalist becomes convinced t...,[18]
8371,Earthquake,Various interconnected people struggle to surv...,[18]
8010,A Field in England,"During the Civil War in 17th-Century England, ...","[36, 27, 53]"


In [85]:
genre_url = "https://api.themoviedb.org/3/genre/movie/list?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US"
genre_response = requests.get(genre_url)
genre_data = genre_response.json()
genre_data['genres']

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 16, 'name': 'Animation'},
 {'id': 35, 'name': 'Comedy'},
 {'id': 80, 'name': 'Crime'},
 {'id': 99, 'name': 'Documentary'},
 {'id': 18, 'name': 'Drama'},
 {'id': 10751, 'name': 'Family'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 36, 'name': 'History'},
 {'id': 27, 'name': 'Horror'},
 {'id': 10402, 'name': 'Music'},
 {'id': 9648, 'name': 'Mystery'},
 {'id': 10749, 'name': 'Romance'},
 {'id': 878, 'name': 'Science Fiction'},
 {'id': 10770, 'name': 'TV Movie'},
 {'id': 53, 'name': 'Thriller'},
 {'id': 10752, 'name': 'War'},
 {'id': 37, 'name': 'Western'}]

In [86]:
genre_name = {}

for genre in genre_data['genres']:
  genre_name[genre['id']] = genre['name']

genre_name[35], genre_name[18], genre_name[10749]

('Comedy', 'Drama', 'Romance')

In [87]:
def get_genre_name(genre_ids):
  names = []
  for name in genre_ids:
    names.append(genre_name[name])
  return names
print(get_genre_name([35, 18, 10749]))

['Comedy', 'Drama', 'Romance']


In [88]:
df['genre'] = df['genre_id'].apply(get_genre_name)
df.head(10)

,movie_name,description,genre_id,genre
0,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","[16, 28, 12, 18]","[Animation, Action, Adventure, Drama]"
1,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","[16, 12, 14, 28]","[Animation, Adventure, Fantasy, Action]"
2,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[Adventure, Animation, Family, Fantasy]"
3,Accidental Partners,Two women discover they were both scammed by t...,"[35, 10749]","[Comedy, Romance]"
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"[16, 28, 14]","[Animation, Action, Fantasy]"
5,Facing El Chapo,Two Mexican officers must survive the final ho...,"[80, 28, 53]","[Crime, Action, Thriller]"
6,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]","[Drama, Crime]"
7,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]","[Drama, Crime]"
8,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[878, 12]","[Science Fiction, Adventure]"
9,Young Hearts,14-year-old Elias increasingly feels like an o...,"[18, 10749]","[Drama, Romance]"


In [89]:
df.to_csv('movies_dataset.csv', index=False)


In [90]:
df['description'][0].lower()
df['description'].str.lower()

,description
0,"arkham asylum has been destroyed, and all its ..."
1,"avatar aang, the world's last airbender, learn..."
2,"a small woodland creature and a majestic bird,..."
3,two women discover they were both scammed by t...
4,the demon slayer corps are drawn into the infi...
...,...
9415,"in an alternate world, humanity and vampires h..."
9416,when he-man is driven from eternia after skele...
9417,"orphaned heiress, english aristocrat and intre..."
9418,"nell, simon, and their boy art are ready to we..."


In [91]:
df['clean description'] = df['description'].str.lower()
df.head()


,movie_name,description,genre_id,genre,clean description
0,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","[16, 28, 12, 18]","[Animation, Action, Adventure, Drama]","arkham asylum has been destroyed, and all its ..."
1,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","[16, 12, 14, 28]","[Animation, Adventure, Fantasy, Action]","avatar aang, the world's last airbender, learn..."
2,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[Adventure, Animation, Family, Fantasy]","a small woodland creature and a majestic bird,..."
3,Accidental Partners,Two women discover they were both scammed by t...,"[35, 10749]","[Comedy, Romance]",two women discover they were both scammed by t...
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"[16, 28, 14]","[Animation, Action, Fantasy]",the demon slayer corps are drawn into the infi...


In [92]:
import string
string.punctuation
exclude = string.punctuation
def remove_punc(descriptions):
  for char in exclude:
    descriptions = descriptions.replace(char, '')
  return descriptions
df['clean description'] = df['clean description'].apply(remove_punc)
df.head()

,movie_name,description,genre_id,genre,clean description
0,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","[16, 28, 12, 18]","[Animation, Action, Adventure, Drama]",arkham asylum has been destroyed and all its i...
1,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","[16, 12, 14, 28]","[Animation, Adventure, Fantasy, Action]",avatar aang the worlds last airbender learns o...
2,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[Adventure, Animation, Family, Fantasy]",a small woodland creature and a majestic bird ...
3,Accidental Partners,Two women discover they were both scammed by t...,"[35, 10749]","[Comedy, Romance]",two women discover they were both scammed by t...
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"[16, 28, 14]","[Animation, Action, Fantasy]",the demon slayer corps are drawn into the infi...


In [93]:
slang_dict = {}
with open("/content/slang.txt",'r') as file:
  for line in file:
    line = line.strip()
    if not line:
      continue
    key,value = line.split('=', 1)
    slang_dict[key.strip()] = value.strip()
print(slang_dict)
print(len( slang_dict))
print(slang_dict["GOAT"])
print(slang_dict["IDK"])


{'A3': 'Anytime, Anywhere, Anyplace', 'ADIH': 'Another Day In Hell', 'AFK': 'Away From Keyboard', 'AFAIK': 'As Far As I Know', 'ASAP': 'As Soon As Possible', 'ASL': 'Age, Sex, Location', 'ATK': 'At The Keyboard', 'ATM': 'At The Moment', 'BAE': 'Before Anyone Else', 'BAK': 'Back At Keyboard', 'BBL': 'Be Back Later', 'BBS': 'Be Back Soon', 'BFN': 'Bye For Now', 'B4N': 'Bye For Now', 'BRB': 'Be Right Back', 'BRUH': 'Bro', 'BRT': 'Be Right There', 'BSAAW': 'Big Smile And A Wink', 'BTW': 'By The Way', 'BWL': 'Bursting With Laughter', 'CSL': 'Can’t Stop Laughing', 'CU': 'See You', 'CUL8R': 'See You Later', 'CYA': 'See You', 'DM': 'Direct Message', 'FAQ': 'Frequently Asked Questions', 'FC': 'Fingers Crossed', 'FIMH': 'Forever In My Heart', 'FOMO': 'Fear Of Missing Out', 'FR': 'For Real', 'FWIW': "For What It's Worth", 'FYP': 'For You Page', 'FYI': 'For Your Information', 'G9': 'Genius', 'GAL': 'Get A Life', 'GG': 'Good Game', 'GMTA': 'Great Minds Think Alike', 'GN': 'Good Night', 'GOAT': 'Gre

In [94]:
def remove_slang(text):
  new_text = []
  for word in text.split():
    if word.upper() in slang_dict:
      new_text.append(slang_dict[word.upper()])
    else:
      new_text.append(word)
  return " ".join(new_text)
remove_slang("I'm a good guy")
df['clean description'] = df['clean description'].apply(remove_slang)
df.head()

,movie_name,description,genre_id,genre,clean description
0,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","[16, 28, 12, 18]","[Animation, Action, Adventure, Drama]",arkham asylum has been destroyed and all its i...
1,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","[16, 12, 14, 28]","[Animation, Adventure, Fantasy, Action]",avatar aang the worlds last airbender learns o...
2,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[Adventure, Animation, Family, Fantasy]",a small woodland creature and a majestic bird ...
3,Accidental Partners,Two women discover they were both scammed by t...,"[35, 10749]","[Comedy, Romance]",two women discover they were both scammed by t...
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"[16, 28, 14]","[Animation, Action, Fantasy]",the demon slayer corps are drawn into the infi...


In [95]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [96]:
def stop_words(text):
  new_text = []
  for word in text.split():
    if word in stopwords.words('english'):
      continue
    else:
      new_text.append(word)
  return " ".join(new_text)
df['clean description'] = df['clean description'].apply(stop_words)
df.head()

,movie_name,description,genre_id,genre,clean description
0,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","[16, 28, 12, 18]","[Animation, Action, Adventure, Drama]",arkham asylum destroyed inmates unleashed upon...
1,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","[16, 12, 14, 28]","[Animation, Adventure, Fantasy, Action]",avatar aang worlds last airbender learns ancie...
2,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[Adventure, Animation, Family, Fantasy]",small woodland creature majestic bird two natu...
3,Accidental Partners,Two women discover they were both scammed by t...,"[35, 10749]","[Comedy, Romance]",two women discover scammed man also got pregna...
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"[16, 28, 14]","[Animation, Action, Fantasy]",demon slayer corps drawn infinity castle tanji...


In [97]:
# from nltk.stem import PorterStemmer
# ps = PorterStemmer()
# type(df['clean description'].iloc[0])
# def stem_words(text):
#   return " ".join([ps.stem(word) for word in text.split()])
# df['clean description'] = df['clean description'].apply(stem_words)
# df.head()


In [98]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
def lemmatize_words(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])
df['clean description'] = df['clean description'].apply(lemmatize_words)
df.head()

,movie_name,description,genre_id,genre,clean description
0,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","[16, 28, 12, 18]","[Animation, Action, Adventure, Drama]",arkham asylum destroyed inmate unleashed upon ...
1,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","[16, 12, 14, 28]","[Animation, Adventure, Fantasy, Action]",avatar aang world last airbender learns ancien...
2,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[Adventure, Animation, Family, Fantasy]",small woodland creature majestic bird two natu...
3,Accidental Partners,Two women discover they were both scammed by t...,"[35, 10749]","[Comedy, Romance]",two woman discover scammed man also got pregna...
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"[16, 28, 14]","[Animation, Action, Fantasy]",demon slayer corp drawn infinity castle tanjir...


In [99]:
from nltk import sent_tokenize, word_tokenize
for sent in sent_tokenize(df['clean description'][0]):
  print(word_tokenize(sent))
  print(sent)
df['clean description']= df['clean description'].apply(sent_tokenize)
df.head()

['arkham', 'asylum', 'destroyed', 'inmate', 'unleashed', 'upon', 'gotham', 'city', 'batman', 'race', 'round', 'greatest', 'enemy', 'pushed', 'physical', 'mental', 'limit', 'final', 'confrontation', 'new', 'threat', 'man', 'called', 'bane']
arkham asylum destroyed inmate unleashed upon gotham city batman race round greatest enemy pushed physical mental limit final confrontation new threat man called bane


,movie_name,description,genre_id,genre,clean description
0,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","[16, 28, 12, 18]","[Animation, Action, Adventure, Drama]",[arkham asylum destroyed inmate unleashed upon...
1,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","[16, 12, 14, 28]","[Animation, Adventure, Fantasy, Action]",[avatar aang world last airbender learns ancie...
2,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[Adventure, Animation, Family, Fantasy]",[small woodland creature majestic bird two nat...
3,Accidental Partners,Two women discover they were both scammed by t...,"[35, 10749]","[Comedy, Romance]",[two woman discover scammed man also got pregn...
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"[16, 28, 14]","[Animation, Action, Fantasy]",[demon slayer corp drawn infinity castle tanji...
